# Deduplicación de registros y agregación a sitios

Convierte los 1.405 registros de diligencias de `full_MIG.geojson` en **sitios candidatos**, que son la unidad de decisión del modelo (Sección 3.2 de la tesis).

Protocolo:
1. **Coordenadas.** Se reconstruyen desde los campos `lat`/`long` (la geometría del geojson tiene valores de relleno fuera de Colombia), con la misma regla de los notebooks de modelos. Se descartan los registros sin coordenadas válidas y se marcan los que caen a más de 1 km de su municipio.
2. **Tipo de sitio.** Un registro es de *cementerio* si su tipo de disposición es `Cementerio`, o si su tipo de disposición está vacío o es `Sin información` y el detalle del lugar menciona cementerio, osario, pabellón o bóveda (los registros marcados como fosa clandestina no se reclasifican). Los cementerios se agrupan por separado, porque son lugares conocidos donde la búsqueda consiste en identificar cuerpos inhumados como NN, y no en localizar un lugar de disposición.
3. **Agrupación espacial.** Dentro de cada municipio y tipo de sitio, dos registros pertenecen al mismo sitio si están a menos de `UMBRAL_M` = 50 m (enlace simple: componentes conexas). El umbral corresponde a la resolución de una coordenada registrada en segundos enteros (~30 m) más el error de un GPS de mano. Se reporta la sensibilidad a 25 y 100 m.
4. **Resultado del sitio.** $D_i = 1$ si el sitio tuvo al menos una exhumación con hallazgo; $D_i = 0$ si solo tuvo prospecciones sin hallazgo; sin resultado si solo tuvo inspecciones, recorridos u otras diligencias (sitios caracterizados pero no intervenidos).
5. **Atributos.** Número de registros, de exhumaciones con hallazgo (aproximación al número de cuerpos recuperados), de prospecciones sin hallazgo y de visitas (fechas distintas), años de la primera y la última diligencia, y si hubo una prospección sin hallazgo seguida de una exhumación con hallazgo (evidencia de detección imperfecta).


In [ ]:
import json
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path
from scipy.sparse.csgraph import connected_components
from scipy.spatial import cKDTree

PATH_PROCESSED  = Path("../data/Processed/")
PATH_MUNICIPIOS = "../data/Municipios_Diciembre_2025_shp/Munpio.shp"
CRS = "EPSG:4326"
CRS_METROS = "EPSG:9377"  # MAGNA-SIRGAS origen nacional

UMBRAL_M = 50
UMBRALES_SENSIBILIDAD = [25, 50, 100]
PATRON_CEMENTERIO = r"CEMENTERIO|OSARIO|PABELL|B[OÓ]VEDA|NECR[OÓ]POLIS"
MESES = {'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4, 'mayo': 5, 'junio': 6, 'julio': 7,
         'agosto': 8, 'septiembre': 9, 'setiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12}


In [ ]:
# ============================================================
# 1. COORDENADAS
# ============================================================
def parsear_coordenada(valor):
    """Convierte decimal o grados-minutos-segundos a decimal (misma regla de los notebooks de modelos)."""
    if valor is None or str(valor).strip() in ['', 'SD', 'nan']:
        return None
    valor = str(valor).strip()
    try:
        return float(valor)
    except ValueError:
        pass
    match = re.search(r"([NSns]?)\s*(\d+)[°]\s*(\d+)['']\s*([\d,.]+)[\"″]?\s*([EWew]?)", valor)
    if match:
        decimal = float(match.group(2)) + float(match.group(3)) / 60 + float(match.group(4).replace(',', '.')) / 3600
        if (match.group(1) or match.group(5)).upper() in ['S', 'W']:
            decimal = -decimal
        return decimal
    return None

def corregir_fila(row):
    es_lat = lambda v: v is not None and -5 <= v <= 13
    es_lon = lambda v: v is not None and -80 <= v <= -66
    lat, lon = parsear_coordenada(row.get('lat')), parsear_coordenada(row.get('long'))
    if es_lat(lat) and es_lon(lon):
        return lat, lon
    if es_lon(lat) and es_lat(lon):
        return lon, lat
    try:
        lat, lon = float(row.get('lat_dd', np.nan)), float(row.get('lon_dd', np.nan))
        if es_lat(lat) and es_lon(lon):
            return lat, lon
        if es_lon(lat) and es_lat(lon):
            return lon, lat
    except (TypeError, ValueError):
        pass
    return None, None

with open(PATH_PROCESSED / "full_MIG.geojson") as f:
    reg = pd.DataFrame([feat['properties'] for feat in json.load(f)['features']])
reg['Cod_Mun'] = reg['Cod_Mun'].astype(str).str.zfill(5)
reg[['lat_c', 'lon_c']] = reg.apply(corregir_fila, axis=1, result_type='expand')
n_total = len(reg)
reg = reg.dropna(subset=['lat_c']).copy()
print(f"Registros: {n_total}  |  con coordenadas válidas: {len(reg)}  |  descartados: {n_total - len(reg)}")

# Distancia de cada registro a su municipio
municipios = gpd.read_file(PATH_MUNICIPIOS)
municipios['MpCodigo'] = municipios['MpCodigo'].astype(str).str.zfill(5)
municipios = municipios.to_crs(CRS_METROS).set_index('MpCodigo')
reg = gpd.GeoDataFrame(reg, geometry=gpd.points_from_xy(reg.lon_c, reg.lat_c), crs=CRS).to_crs(CRS_METROS)
reg['dist_municipio_m'] = [municipios.geometry[c].distance(p) if c in municipios.index else np.nan
                           for c, p in zip(reg['Cod_Mun'], reg.geometry)]
reg['fuera_municipio'] = ~(reg['dist_municipio_m'] <= 1000)
print(f"Registros a más de 1 km de su municipio (o sin municipio): {reg['fuera_municipio'].sum()}")
print(reg.loc[reg['fuera_municipio'], 'Cod_Mun'].value_counts().to_string())


In [ ]:
# ============================================================
# 2. TIPO DE SITIO Y RESULTADO DE CADA REGISTRO
# ============================================================
texto = (reg['Detalle del lugar'].fillna('') + ' ' + reg['Observaciones geográficas'].fillna('')).str.upper()
disposicion = reg['Tipo disposición (Cuando aplique)']
sin_disposicion = disposicion.isna() | (disposicion == 'Sin información')
reg['cementerio'] = (disposicion == 'Cementerio') | (sin_disposicion & texto.str.contains(PATRON_CEMENTERIO))
print(f"Cementerio por tipo de disposición: {(disposicion == 'Cementerio').sum()}  |  "
      f"por texto sin disposición: {(sin_disposicion & texto.str.contains(PATRON_CEMENTERIO)).sum()}")
reg['tipo_sitio'] = np.where(reg['cementerio'], 'cementerio', 'otro')

reg['anio'] = pd.to_numeric(reg['Año de la diligencia'], errors='coerce')
reg['mes'] = pd.to_numeric(reg['Mes de la diligencia'], errors='coerce')
mes_texto = reg['Mes de la diligencia'].astype(str).str.strip().str.lower().map(MESES)
reg['mes'] = reg['mes'].fillna(mes_texto)
reg['fecha'] = reg['anio'] + (reg['mes'].fillna(6) - 1) / 12  # mes desconocido: mitad del año

print(pd.crosstab(reg['tipo_sitio'], reg['label'].fillna(-1).map({1: 'hallazgo', 0: 'sin hallazgo', -1: 'sin resultado'})))


In [ ]:
# ============================================================
# 3. AGRUPACIÓN ESPACIAL EN SITIOS
# ============================================================
def agrupar(reg, umbral_m):
    """Componentes conexas de registros a menos de umbral_m, por municipio y tipo de sitio."""
    sitio = pd.Series(index=reg.index, dtype=object)
    xy = np.c_[reg.geometry.x, reg.geometry.y]
    pos = {idx: i for i, idx in enumerate(reg.index)}
    for (mun, tipo), g in reg.groupby(['Cod_Mun', 'tipo_sitio']):
        filas = [pos[i] for i in g.index]
        pares = cKDTree(xy[filas]).query_pairs(umbral_m, output_type='ndarray')
        n = len(filas)
        if len(pares):
            from scipy.sparse import coo_matrix
            adj = coo_matrix((np.ones(len(pares)), (pares[:, 0], pares[:, 1])), shape=(n, n))
            _, comp = connected_components(adj, directed=False)
        else:
            comp = np.arange(n)
        sitio[g.index] = [f"{mun}_{tipo[0].upper()}{c:03d}" for c in comp]
    return sitio

def resumir_sitios(reg):
    def resumen(g):
        hall = g['label'] == 1
        neg = g['label'] == 0
        pos_hall = g.loc[hall, 'fecha'].min()
        return pd.Series({
            'municipio': g['Cod_Mun'].iloc[0],
            'tipo_sitio': g['tipo_sitio'].iloc[0],
            'lon': g['lon_c'].mean(),
            'lat': g['lat_c'].mean(),
            'n_registros': len(g),
            'n_exhumaciones_hallazgo': int(hall.sum()),
            'n_prospecciones_sin_hallazgo': int(neg.sum()),
            'n_otras_diligencias': int(g['label'].isna().sum()),
            'n_visitas': g[['anio', 'mes']].drop_duplicates().dropna(subset=['anio']).shape[0],
            'anio_primera': g['anio'].min(),
            'anio_ultima': g['anio'].max(),
            'disposicion': g['Tipo disposición (Cuando aplique)'].mode().iloc[0]
                           if g['Tipo disposición (Cuando aplique)'].notna().any() else None,
            'fuera_municipio': bool(g['fuera_municipio'].all()),
            'D': 1.0 if hall.any() else (0.0 if neg.any() else np.nan),
            'negativa_previa_a_hallazgo': bool(hall.any() and neg.any()
                                               and (g.loc[neg, 'fecha'] < pos_hall).any()),
        })
    return reg.groupby('sitio').apply(resumen)

reg['sitio'] = agrupar(reg, UMBRAL_M)
sitios = resumir_sitios(reg)
print(f"Sitios con umbral de {UMBRAL_M} m: {len(sitios)}")


In [ ]:
# ============================================================
# 4. RESUMEN
# ============================================================
def tabla_resumen(sitios):
    filas = []
    for tipo, g in [('Todos', sitios), *sitios.groupby('tipo_sitio')]:
        filas.append({'tipo': tipo, 'sitios': len(g),
                      'con hallazgo (D=1)': int((g['D'] == 1).sum()),
                      'sin hallazgo (D=0)': int((g['D'] == 0).sum()),
                      'sin resultado': int(g['D'].isna().sum()),
                      'tasa de hallazgo': round(g['D'].mean(), 3),
                      'registros por sitio': round(g['n_registros'].mean(), 2),
                      'municipios': g['municipio'].nunique()})
    return pd.DataFrame(filas)

print(tabla_resumen(sitios).to_string(index=False))

print(f"\nSitios con prospección sin hallazgo seguida de exhumación con hallazgo: "
      f"{sitios['negativa_previa_a_hallazgo'].sum()}")
print(f"Sitios fuera de su municipio: {sitios['fuera_municipio'].sum()}")
print(f"\nDistribución de registros por sitio:")
print(sitios['n_registros'].describe().round(2).to_string())


In [ ]:
# ============================================================
# 5. SENSIBILIDAD AL UMBRAL
# ============================================================
filas = []
for umbral in UMBRALES_SENSIBILIDAD:
    r = reg.copy()
    r['sitio'] = agrupar(r, umbral)
    s = resumir_sitios(r)
    otros = s[s['tipo_sitio'] == 'otro']
    filas.append({'umbral_m': umbral, 'sitios': len(s),
                  'sitios no cementerio': len(otros),
                  'no cementerio con resultado': int(otros['D'].notna().sum()),
                  'tasa de hallazgo no cementerio': round(otros['D'].mean(), 3)})
print(pd.DataFrame(filas).to_string(index=False))


In [ ]:
# ============================================================
# 6. GUARDAR
# ============================================================
sitios.reset_index().to_csv(PATH_PROCESSED / "sitios.csv", index=False)
reg[['COD_REF_EQ', 'sitio', 'Cod_Mun', 'tipo_sitio', 'Tipo de diligencia', 'label',
     'anio', 'mes', 'lon_c', 'lat_c', 'fuera_municipio']].to_csv(
    PATH_PROCESSED / "registros_a_sitios.csv", index=False)
print(f"✓ sitios.csv ({len(sitios)} sitios) y registros_a_sitios.csv ({len(reg)} registros)")
